# Notebook 06: Advanced Agent Patterns

本实验探索高级 Agent 模式：Tool Use、对话记忆、流式处理。

## 前置条件
1. Neo4j 正在运行并已加载种子数据
2. `.env` 中配置了 LLM Provider

## 学习内容
- Tool 定义和手动调用
- ToolCallingAgent vs SmartHomeAgent 对比
- 对话记忆的多轮交互
- Streaming 对比

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

print("Environment loaded.")

## 1. Tool 定义和手动调用

先了解有哪些 Tool 可用，然后手动调用它们。

In [ ]:
from src.agent.tools import get_all_tools

tools = get_all_tools()

print("=== 可用 Tools ===")
for tool in tools:
    print(f"\n{tool.name}:")
    print(f"  描述: {tool.description[:100]}...")
    print(f"  参数: {tool.args}")

In [ ]:
# 手动调用 Tool：查询房间设备
result = tools[0].invoke({"room_name": "living"})
print("=== query_room_devices('living') ===")
print(result)

In [ ]:
# 手动调用 Tool：搜索场景
result = tools[2].invoke({"keyword": "movie"})
print("=== search_scenes('movie') ===")
print(result)

In [ ]:
# 手动调用 Tool：执行设备命令（模拟）
result = tools[3].invoke({
    "device_id": "living_ceiling_01",
    "capability": "dim",
    "value": "20",
})
print("=== execute_device_command ===")
print(result)

## 2. ToolCallingAgent vs SmartHomeAgent

对比两种 Agent 处理相同请求的方式。

- **SmartHomeAgent**: 显式工作流（预定义的节点和边）
- **ToolCallingAgent**: 工具调用（LLM 动态选择工具）

In [ ]:
from src.agent.workflow import SmartHomeAgent
from src.agent.tool_agent import ToolCallingAgent

# 初始化两个 Agent
explicit_agent = SmartHomeAgent()
tool_agent = ToolCallingAgent()

print("两个 Agent 已初始化。")

In [ ]:
# 测试请求
test_message = "打开客厅的灯"

print("=" * 60)
print(f"测试消息: {test_message}")
print("=" * 60)

# --- 显式工作流 ---
import time

print("\n--- SmartHomeAgent (显式工作流) ---")
start = time.time()
response1, trace1 = explicit_agent.run_with_trace(test_message)
duration1 = time.time() - start

print(f"响应: {response1[:200]}")
print(f"推理步骤: {len(trace1)}")
for step in trace1:
    print(f"  - {step}")
print(f"耗时: {duration1:.2f}s")

In [ ]:
# --- 工具调用 ---
print("--- ToolCallingAgent (工具调用) ---")
start = time.time()
response2, trace2 = tool_agent.run_with_trace(test_message)
duration2 = time.time() - start

print(f"响应: {response2[:200]}")
print(f"推理步骤: {len(trace2)}")
for step in trace2:
    print(f"  - {step}")
print(f"耗时: {duration2:.2f}s")

In [ ]:
# 对比总结
print("\n=== 对比总结 ===")
print(f"{'指标':<20} {'显式工作流':<20} {'工具调用':<20}")
print("-" * 60)
print(f"{'响应长度':<20} {len(response1):<20} {len(response2):<20}")
print(f"{'推理步骤':<20} {len(trace1):<20} {len(trace2):<20}")
print(f"{'耗时':<20} {duration1:.2f}s{'':<15} {duration2:.2f}s")

## 3. 对话记忆

测试 ToolCallingAgent 的多轮对话能力。

In [ ]:
from src.agent.memory import ConversationMemory

# 创建带记忆的 Agent
memory = ConversationMemory(max_turns=5)
agent_with_memory = ToolCallingAgent(memory=memory)

session_id = "demo-session-1"

print("=== 多轮对话测试 ===")

In [ ]:
# 第一轮
r1 = agent_with_memory.run("客厅有什么设备？", session_id=session_id)
print(f"用户: 客厅有什么设备？")
print(f"助手: {r1}")
print(f"\n记忆长度: {memory.get_session_length(session_id)} 条消息")

In [ ]:
# 第二轮 — 引用上文
r2 = agent_with_memory.run("把灯调暗一点", session_id=session_id)
print(f"用户: 把灯调暗一点")
print(f"助手: {r2}")
print(f"\n记忆长度: {memory.get_session_length(session_id)} 条消息")

In [ ]:
# 查看完整对话历史
print("=== 对话历史 ===")
print(memory.get_context_string(session_id))

## 4. Streaming 对比

观察两种 Agent 的流式输出差异。

In [ ]:
# SmartHomeAgent 流式
print("=== SmartHomeAgent 流式 ===")
for event in explicit_agent.run_streaming("电影模式"):
    for node_name, data in event.items():
        keys = list(data.keys())
        print(f"  [{node_name}] → 输出字段: {keys}")

In [ ]:
# ToolCallingAgent 流式
print("=== ToolCallingAgent 流式 ===")
for event in tool_agent.run_streaming("电影模式"):
    for node_name, data in event.items():
        messages = data.get('messages', [])
        if messages:
            last_msg = messages[-1]
            msg_type = type(last_msg).__name__
            content_preview = str(last_msg.content)[:80] if hasattr(last_msg, 'content') else 'N/A'
            print(f"  [{node_name}] → {msg_type}: {content_preview}...")

## 5. 设计模式总结

| 维度 | 显式工作流 | 工具调用 |
|------|-----------|----------|
| 流程控制 | 预定义状态图 | LLM 动态决定 |
| 可预测性 | 高 | 低 |
| 灵活性 | 低 | 高 |
| 调试 | 容易 | 较难 |
| LLM 调用 | 固定次数 | 不固定 |
| 适用场景 | 流程明确 | 开放域 |

**实际建议:** 从显式工作流开始，在需要灵活性时引入工具调用。

In [ ]:
print("实验完成！")
print("\n关键收获:")
print("1. Tool Use 让 LLM 能够调用外部工具")
print("2. 对话记忆使多轮交互成为可能")
print("3. 显式工作流和工具调用各有优劣")
print("4. 流式处理提升用户体验")